<a href="https://colab.research.google.com/github/MUHAMMADAFFAN786/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUHAMMADAFFAN786/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Ranked actions + reason codes

The goal of this section is to turn the validated model output into a ranked content-action queue.

The ranking is used as decision-support for human review. It does not automatically publish, delete, or change content.

Each recommended action has a reason code so that the reviewer can understand why the item was placed in the queue.

Reason codes:

- HIGH_SCORE — stronger observed model signal and higher review priority.
- MEDIUM_SCORE — moderate observed model signal and normal review priority.
- LOW_SCORE — weaker observed model signal and lower review priority.
- REFRESH_SIGNAL — the available evidence suggests that the content may be worth reviewing for a refresh.

The ranking should be treated as directional rather than as a guaranteed prediction of content performance.

A human reviewer should check the content and context before taking any action.

In [59]:
# ML-10 — Step 1: Ranked actions + reason codes

import os
import pandas as pd

print("=== ML-10: Ranked Action Queue ===")

queue_path = "outputs/refresh_queue_sample.csv"

# Load actual Week-5 output
if not os.path.exists(queue_path):
    raise FileNotFoundError(
        "Week-5 refresh_queue_sample.csv was not found."
    )

queue = pd.read_csv(queue_path)

print("Loaded Week-5 output successfully.")
print("Rows:", len(queue))
print("Columns:", len(queue.columns))
print("\nColumn names:")
print(list(queue.columns))

# ---------------------------------------------------------
# Detect useful columns from the actual output
# ---------------------------------------------------------

priority_col = next(
    (c for c in queue.columns if "priority" in c.lower()),
    None
)

action_col = next(
    (c for c in queue.columns if "action" in c.lower()),
    None
)

score_candidates = [
    c for c in queue.columns
    if "score" in c.lower()
    and pd.api.types.is_numeric_dtype(queue[c])
]

score_col = score_candidates[0] if score_candidates else None

# ---------------------------------------------------------
# Create ranking from actual model output
# ---------------------------------------------------------

if priority_col is not None:

    priority_order = {
        "high": 0,
        "medium": 1,
        "low": 2
    }

    queue["_priority_order"] = (
        queue[priority_col]
        .astype(str)
        .str.lower()
        .map(priority_order)
        .fillna(3)
    )

else:
    queue["_priority_order"] = 3


if score_col is not None:

    queue = queue.sort_values(
        by=["_priority_order", score_col],
        ascending=[True, False]
    )

else:

    queue = queue.sort_values(
        by="_priority_order"
    )


queue = queue.reset_index(drop=True)

# Add final rank
queue.insert(0, "rank", range(1, len(queue) + 1))

# ---------------------------------------------------------
# Reason codes
# ---------------------------------------------------------

def make_reason_code(row):

    priority = str(
        row.get(priority_col, "")
    ).lower()

    action = str(
        row.get(action_col, "")
    ).lower()

    if "refresh" in action:
        return "REFRESH_SIGNAL"

    if priority == "high":
        return "HIGH_PRIORITY"

    if priority == "medium":
        return "MEDIUM_PRIORITY"

    return "LOW_PRIORITY"


queue["reason_code"] = queue.apply(
    make_reason_code,
    axis=1
)

queue["human_review"] = "Required"

# Remove temporary sorting column
queue = queue.drop(
    columns=["_priority_order"]
)

# ---------------------------------------------------------
# Display ranked queue
# ---------------------------------------------------------

print("\n=== FINAL RANKED ACTION QUEUE ===")

display(queue.head(10))

print("\nRanking basis:")

if priority_col:
    print("Priority column:", priority_col)

if score_col:
    print("Model score column:", score_col)

if action_col:
    print("Action column:", action_col)

print("\nReason codes:")
print(queue["reason_code"].value_counts())

print("\nHuman review is required before any action is taken.")

=== ML-10: Ranked Action Queue ===
Loaded Week-5 output successfully.
Rows: 200
Columns: 28

Column names:
['final_rank', 'content_id', 'client_id', 'final_refresh_score', 'best_model_name', 'best_model_probability', 'baseline_refresh_score', 'confidence', 'suggested_action', 'final_reason_codes', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

=== FINAL RANKED ACTION QUEUE ===


,rank,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,...,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,reason_code,human_review
0,1,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,...,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
1,2,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,...,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
2,3,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,...,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking,REFRESH_SIGNAL,Required
3,4,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,...,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
4,5,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,...,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
5,6,6,content_b69288c5e701,client_3fdba35f04,80.754770,random_forest,0.795713,0.787358,high,refresh_and_review_ctr,...,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
6,7,7,content_9b6df29f7889,client_3fdba35f04,80.632923,random_forest,0.846245,0.673530,high,refresh_and_review_ctr,...,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,moderate,page_1,REFRESH_SIGNAL,Required
7,8,8,content_bb6ebb5ec8c8,client_3fdba35f04,80.371236,random_forest,0.834638,0.690665,high,refresh_and_review_ctr,...,LOW,keyword article,informational,91-180,91-180,1000-2000,moderate,striking,REFRESH_SIGNAL,Required
8,9,9,content_4d76cdb3387b,client_3fdba35f04,80.362748,random_forest,0.843092,0.671993,medium,refresh_and_review_ctr,...,HIGH,keyword article,commercial,91-180,91-180,1000-2000,moderate,top_3,REFRESH_SIGNAL,Required
9,10,10,content_b4f35d640b1c,client_3fdba35f04,80.321757,random_forest,0.843803,0.669168,medium,refresh,...,HIGH,keyword article,commercial,91-180,91-180,1000-2000,good,page_3_5,REFRESH_SIGNAL,Required



Ranking basis:
Model score column: final_refresh_score
Action column: suggested_action

Reason codes:
reason_code
REFRESH_SIGNAL    200
Name: count, dtype: int64

Human review is required before any action is taken.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [60]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This playbook is intended for content or marketing reviewers who need to prioritize which content items should be reviewed first. It uses the validated model output to provide a ranked list of items that may need attention.

The output is decision-support, not an automatic decision. A higher model score does not guarantee that an action will improve performance. The recommendations are limited by the available features, historical data, validation design, and model performance. Results may be less reliable when new content or situations differ substantially from the data used to build and validate the model.

A human should review the recommendation and its reason code before taking any action.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [61]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Human review is required before any recommended action is taken. The reviewer should check the item's current performance, content quality, relevance, reason code, and whether the recommendation still makes sense in the current context.

The model should not automatically publish, delete, rewrite, or make major changes to content. It should also not make irreversible business decisions or decisions involving sensitive information. Low-confidence, unusual, or unclear recommendations should be sent for manual review.

The model is used to prioritize review, not to replace human judgment.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [62]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The recommendations should be monitored over time to check whether they remain useful. Warning signs include a decline in model performance, changes in the distribution of the input features, increasing disagreement between recommendations and human review, or major changes in content behaviour.

A retraining review should be considered when new validated data becomes available, when model performance consistently declines, or when the underlying content environment changes substantially. A single unusual result should not be enough to trigger retraining; the decision should be based on repeated measured evidence.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [63]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The ranked action queue will be exported to work/outputs/ so that the research paper can reuse the same validated output. The export should preserve the ranked items, model score, recommended action, and reason code where available.

The notebook should regenerate the queue from the validated model output rather than relying on manually edited files. This keeps the paper's recommendations traceable to the notebook results.

In [64]:
import os
import pandas as pd

os.makedirs("work/outputs", exist_ok=True)

# Find the ranked queue created in earlier sections
queue_df = None

for name in ["ranked_queue", "action_queue", "queue", "df"]:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        queue_df = globals()[name].copy()
        print(f"Using DataFrame: {name}")
        break

if queue_df is not None:
    output_path = "work/outputs/ranked_action_queue.csv"
    queue_df.to_csv(output_path, index=False)

    print("Export completed successfully.")
    print("File:", output_path)
    print("Rows:", len(queue_df))
    print("Columns:", len(queue_df.columns))
    display(queue_df.head(10))
else:
    print("No ranked action queue found.")
    print("Run Section 1 first and then run this cell.")

Using DataFrame: queue
Export completed successfully.
File: work/outputs/ranked_action_queue.csv
Rows: 200
Columns: 31


,rank,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,...,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,reason_code,human_review
0,1,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,...,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
1,2,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,...,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
2,3,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,...,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking,REFRESH_SIGNAL,Required
3,4,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,...,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
4,5,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,...,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
5,6,6,content_b69288c5e701,client_3fdba35f04,80.754770,random_forest,0.795713,0.787358,high,refresh_and_review_ctr,...,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1,REFRESH_SIGNAL,Required
6,7,7,content_9b6df29f7889,client_3fdba35f04,80.632923,random_forest,0.846245,0.673530,high,refresh_and_review_ctr,...,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,moderate,page_1,REFRESH_SIGNAL,Required
7,8,8,content_bb6ebb5ec8c8,client_3fdba35f04,80.371236,random_forest,0.834638,0.690665,high,refresh_and_review_ctr,...,LOW,keyword article,informational,91-180,91-180,1000-2000,moderate,striking,REFRESH_SIGNAL,Required
8,9,9,content_4d76cdb3387b,client_3fdba35f04,80.362748,random_forest,0.843092,0.671993,medium,refresh_and_review_ctr,...,HIGH,keyword article,commercial,91-180,91-180,1000-2000,moderate,top_3,REFRESH_SIGNAL,Required
9,10,10,content_b4f35d640b1c,client_3fdba35f04,80.321757,random_forest,0.843803,0.669168,medium,refresh,...,HIGH,keyword article,commercial,91-180,91-180,1000-2000,good,page_3_5,REFRESH_SIGNAL,Required


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 6. Self-check

- [x] Ranked actions and reason codes are included.
- [x] Intended use and limitations are clearly stated.
- [x] Human review requirements are defined.
- [x] No-go cases are identified.
- [x] Monitoring and retraining triggers are documented.
- [x] The ranked action queue is exported for reuse in the research paper.
- [x] The recommendations are framed as decision-support rather than automatic production decisions.
- [x] The notebook uses the validated Week-5 output rather than invented model results.